# UCS420: Cognitive Computing
# Assignment 4 - A Cognitive FAQ System Using Pandas
# Roll Number: 1024170401

## Q1: Build Your Personalized Knowledge Base

In [ ]:
import pandas as pd

roll_number = "1024170401"
last_two_digits = roll_number[-2:]
print(f"Roll Number: {roll_number}")
print(f"Last Two Digits: {last_two_digits}")

digit_1, digit_2 = int(last_two_digits[0]), int(last_two_digits[1])
print(f"Digit 1: {digit_1}, Digit 2: {digit_2}")

categories = ["billing", "account", "general"]
category_1 = categories[digit_1 % 3]
category_2 = categories[digit_2 % 3]
print(f"Category for digit {digit_1}: {category_1}")
print(f"Category for digit {digit_2}: {category_2}")

In [ ]:
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

custom_entries = [
    {"question": "what are the refund policies", 
     "answer": "Refunds are processed within 5-7 business days to your original payment method.",
     "keywords": "refund money back policy return", 
     "category": "billing"},
    {"question": "how do i update my registered email", 
     "answer": "Go to Account Settings > Email > Verify New Email and confirm via OTP.",
     "keywords": "email update change account profile", 
     "category": "account"},
]

all_entries = fixed_entries + custom_entries
df = pd.DataFrame(all_entries)

print("Personalized Knowledge Base (6 FAQ Entries):")
print(df)
print(f"\nDataFrame Shape: {df.shape}")

## Q2: Generate and Score a Hypothesis

In [ ]:
def score_query(query, df):
    query_lower = query.lower().split()
    results = []
    
    for idx, row in df.iterrows():
        question_words = set(row['question'].lower().split())
        keywords_list = set(row['keywords'].lower().split())
        all_words = question_words | keywords_list
        
        matches = sum(1 for word in query_lower if word in all_words)
        if matches > 0:
            confidence_score = matches / len(all_words) if len(all_words) > 0 else 0
            results.append({
                'index': idx,
                'question': row['question'],
                'answer': row['answer'],
                'category': row['category'],
                'matches': matches,
                'confidence': confidence_score
            })
    
    if results:
        results_df = pd.DataFrame(results).sort_values('confidence', ascending=False)
        return results_df
    else:
        return pd.DataFrame()

test_query_1 = "how to reset password"
print(f"Query: '{test_query_1}'")
print(score_query(test_query_1, df))
print("\n" + "="*80 + "\n")

test_query_2 = "fee and payment"
print(f"Query: '{test_query_2}'")
print(score_query(test_query_2, df))

## Q3: Write a Function to Filter by Category

In [ ]:
def same_category(category_name, df):
    filtered_df = df[df['category'] == category_name][['question', 'answer', 'category']]
    return filtered_df

selected_category = "billing"
print(f"All FAQ entries in '{selected_category}' category:")
print(same_category(selected_category, df))

## Q4: Add New Keyword and Save to CSV

In [ ]:
print("Current DataFrame:")
print(df)
print("\n")

entry_to_update = 0
new_keyword = "discount"

print(f"Entry being updated (Index {entry_to_update}):")
print(f"Question: {df.iloc[entry_to_update]['question']}")
print(f"Current Keywords: {df.iloc[entry_to_update]['keywords']}")
print(f"New Keyword to Add: {new_keyword}")

df.loc[entry_to_update, 'keywords'] = df.iloc[entry_to_update]['keywords'] + " " + new_keyword

print(f"Updated Keywords: {df.iloc[entry_to_update]['keywords']}")
print("\n")

csv_filename = f"{roll_number}_faq_data.csv"
df.to_csv(csv_filename, index=False)
print(f"DataFrame saved to '{csv_filename}'")
print("\nSaved CSV Content:")
print(df)

## Q5: GroupBy Category Counts

In [ ]:
category_counts = df.groupby('category').size()
print("FAQ entries per category:")
print(category_counts)
print("\nDetailed breakdown:")
for category, count in category_counts.items():
    print(f"{category}: {count} entries")

## Q6: Enhanced Scoring Function with Tie Detection

In [ ]:
def score_query_with_tie_detection(query, df):
    query_lower = query.lower().split()
    results = []
    
    for idx, row in df.iterrows():
        question_words = set(row['question'].lower().split())
        keywords_list = set(row['keywords'].lower().split())
        all_words = question_words | keywords_list
        
        matches = sum(1 for word in query_lower if word in all_words)
        if matches > 0:
            confidence_score = matches / len(all_words) if len(all_words) > 0 else 0
            results.append({
                'index': idx,
                'question': row['question'],
                'answer': row['answer'],
                'category': row['category'],
                'matches': matches,
                'confidence': confidence_score
            })
    
    if not results:
        print("No matches found.")
        return pd.DataFrame()
    
    results_df = pd.DataFrame(results).sort_values('confidence', ascending=False)
    max_confidence = results_df['confidence'].max()
    tied_entries = results_df[results_df['confidence'] == max_confidence]
    
    if len(tied_entries) > 1:
        print(f"Tie detected! {len(tied_entries)} entries with equal confidence ({max_confidence:.4f}):")
        print(tied_entries[['question', 'answer', 'category', 'confidence']])
    else:
        print(f"Single best match (Confidence: {max_confidence:.4f}):")
        print(tied_entries[['question', 'answer', 'category', 'confidence']])
    
    return results_df

print("DEMONSTRATION 1: Query with TIE")
print("-" * 80)
query_with_tie = "fee payment"
print(f"Query: '{query_with_tie}'")
print()
score_query_with_tie_detection(query_with_tie, df)

print("\n\n" + "="*80 + "\n")
print("DEMONSTRATION 2: Query WITHOUT TIE")
print("-" * 80)
query_without_tie = "password reset"
print(f"Query: '{query_without_tie}'")
print()
score_query_with_tie_detection(query_without_tie, df)